In [1]:
# ─────────────────────────── Planner / Guider Layer ─────────────────────────────
# Requirements: python-dotenv, langchain-google-genai, pandas
# Uses the same LLM setup from your previous notebook cell

import os, json, re, textwrap
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# ─── Config ─────────────────────────────────────────────────────────────────────
METADATA_DIR = Path("metadata")
PLANS_DIR = Path("plans")
PLANS_DIR.mkdir(parents=True, exist_ok=True)

# Reuse your LLM init
load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    api_key=google_api_key,
    temperature=0.2
)

# ─── Utilities: load/find metadata, retrofit metadata_path if missing ───────────
def _metadata_path_from_dataset_file(file_path: str | Path) -> Path:
    """Given a dataset file path, return metadata/<stem>_metadata.json"""
    p = Path(file_path)
    return METADATA_DIR / f"{p.stem}_metadata.json"

def _metadata_path_from_stem(stem: str) -> Path:
    return METADATA_DIR / f"{stem}_metadata.json"

def load_metadata(file_or_stem: str | Path) -> Tuple[Dict[str, Any], Path]:
    """
    Load metadata JSON by:
    - direct path to metadata file, OR
    - dataset file path (we derive metadata path), OR
    - stem name (we look under metadata/).
    Returns (metadata_dict, metadata_json_path)
    """
    p = Path(file_or_stem)
    candidate_paths: List[Path] = []

    if p.suffix.lower() == ".json" and p.exists():
        candidate_paths.append(p)
    elif p.exists() and p.suffix.lower() in {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet", ".pq"}:
        candidate_paths.append(_metadata_path_from_dataset_file(p))
    else:
        # treat as stem
        candidate_paths.append(_metadata_path_from_stem(str(p)))

    for c in candidate_paths:
        if c.exists():
            md = json.loads(Path(c).read_text(encoding="utf-8"))
            # retrofit: ensure metadata includes its own path
            if "dataset" in md:
                ds = md["dataset"]
                changed = False
                if "metadata_path" not in ds:
                    ds["metadata_path"] = str(c.resolve())
                    changed = True
                if changed:
                    c.write_text(json.dumps(md, ensure_ascii=False, indent=2), encoding="utf-8")
            return md, c

    raise FileNotFoundError(f"Could not find metadata for: {file_or_stem}")

# ─── Helper: extract quick lookup from metadata ─────────────────────────────────
def _column_index(metadata: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    cols = metadata.get("columns", [])
    idx = {c["name"]: c for c in cols}
    return idx

def _normalize_key(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "", (s or "")).upper()

def _map_to_actual_columns(requested_cols: List[str], metadata: Dict[str, Any]) -> List[str]:
    """Keep only columns that actually exist (case-insensitive check)."""
    actual = _column_index(metadata)
    norm_map = {_normalize_key(k): k for k in actual.keys()}
    out = []
    for rc in requested_cols or []:
        nk = _normalize_key(rc)
        if nk in norm_map:
            out.append(norm_map[nk])
    # de-dup preserving order
    seen = set()
    final = []
    for c in out:
        if c not in seen:
            final.append(c)
            seen.add(c)
    return final

# ─── SYSTEM PROMPT (edit here to change planner behavior globally) ──────────────
PLANNER_SYSTEM_PROMPT = """
You are a senior analytics planner. You must return STRICT JSON ONLY (no markdown, no prose outside JSON).
Given: (1) dataset metadata, (2) optional user objectives, plan a complete analysis and visualization guide that the code-generation step can execute.

Return a JSON object with these keys:

{
  "dataset": {
    "name": "...",
    "file_path": "...",
    "file_abspath": "...",
    "metadata_path": "..."
  },
  "objectives": [
    {"text": "objective string", "why": "why this objective matters based on metadata"}
    // If user provides none, propose 3-6 high-value objectives for sales analytics
  ],
  "kpis": [
    {
      "id": "snake_case_id",
      "label": "Human label",
      "why": "business justification in 1-2 lines",
      "columns": ["EXACT_COLUMN_NAMES_USED"],
      "derived_fields": [
        {"name": "revenue", "expr": "QUANTITYORDERED * PRICEEACH"} // if needed; else []
      ],
      "grain": "one of: order_line | order | day | week | month | quarter | year | customer | product | geo",
      "filters": [{"expr": "STATUS != 'Cancelled'"}]
    }
  ],
  "analyses": [
    {
      "id": "snake_case_id",
      "label": "What we analyze",
      "why": "why this analysis is useful",
      "group_by": ["COLUMN_NAMES or DERIVED like YEAR_ID, MONTH_ID, PRODUCTLINE"],
      "metrics": [{"name": "sales", "expr": "sum(SALES)"}],
      "filters": [{"expr": "YEAR_ID in (2003,2004,2005)"}],
      "grain": "month"
    }
  ],
  "visualizations": [
    {
      "id": "snake_case_id",
      "label": "Human label",
      "type": "one of: line | bar | horizontal_bar | stacked_bar | heatmap | histogram | box | map",
      "why": "explain why this is the right chart",
      "data_access": {
        "dataset_path": "...",             // where to read data
        "format": "csv|xlsx|parquet",
        "encoding": "utf-8|cp1252|excel|parquet"
      },
      "spec": {
        "x": "COLUMN for x-axis or derived (e.g., MONTH_ID)",
        "y": ["COLUMN(s) for y"],
        "series_by": ["optional series column"],
        "sort": "optional sort expression",
        "stack": true/false,
        "agg": [{"name": "sales", "expr": "sum(SALES)"}],
        "derived_fields": [
          {"name": "MONTH", "expr": "to_period(ORDERDATE, 'M')"} // if needed
        ],
        "filters": [{"expr": "STATUS != 'Cancelled'"}]
      },
      "columns_used": ["EXACT_COLUMN_NAMES_USED"], // must match metadata columns
      "export": {
        "suggested_path": "charts/<name>.html",
        "format": "html|png"
      }
    }
  ],
  "notes": [
    "assumptions, caveats, or data quality warnings that affect analysis choices"
  ]
}

Rules:
- Use ONLY columns that exist in the provided metadata; do not invent or rename columns.
- Prefer SALES if present; otherwise derive revenue from QUANTITYORDERED * PRICEEACH (or mention derivation).
- If ORDERDATE is string but usable for time series, reference YEAR_ID/MONTH_ID if present.
- If geo columns exist (COUNTRY/STATE/CITY), include at least one geo breakdown.
- Aim for 4–8 visualizations covering trends, ranking (top N), composition, and variance.
- Keep explanations concise (<= 2 lines per 'why').
""".strip()

# ─── Build the user payload for the LLM (organized & editable) ──────────────────
def build_planner_user_payload(
    metadata: Dict[str, Any],
    objectives: Optional[List[str]] = None
) -> Dict[str, Any]:
    dataset = metadata.get("dataset", {})
    cols = metadata.get("columns", [])
    # Only pass the fields the model needs to choose correctly
    light_cols = [
        {
            "name": c["name"],
            "dtype": c.get("dtype"),
            "semantics": c.get("semantics", []),
            "null_pct": c.get("null_pct"),
            "unique_count": c.get("unique_count"),
        }
        for c in cols
    ]
    payload = {
        "dataset": {
            "name": dataset.get("name"),
            "file": dataset.get("file"),
            "file_path": dataset.get("file_path"),
            "file_abspath": dataset.get("file_abspath"),
            "file_encoding": dataset.get("file_encoding"),
            "metadata_path": dataset.get("metadata_path"),  # may have been retrofitted
            "row_count": dataset.get("row_count"),
            "column_count": dataset.get("column_count"),
        },
        "columns": light_cols,
        "dataset_description": metadata.get("dataset_description", ""),
        "assumptions": metadata.get("assumptions", []),
        "user_objectives": objectives or []
    }
    return payload

# ─── Call LLM and parse STRICT JSON; then validate columns ──────────────────────
def _parse_strict_json(text: str) -> Dict[str, Any]:
    cleaned = re.sub(r"```(json)?", "", text, flags=re.I).strip()
    return json.loads(cleaned)

def _validate_and_prune_plan(plan: Dict[str, Any], metadata: Dict[str, Any]) -> Dict[str, Any]:
    """Ensure columns_used, spec.y, spec.series_by etc. reference only real columns."""
    col_idx = _column_index(metadata)
    all_cols = list(col_idx.keys())

    # dataset block: ensure metadata_path present
    ds = plan.get("dataset", {}) or {}
    mds = metadata.get("dataset", {})
    ds.setdefault("name", mds.get("name"))
    ds.setdefault("file_path", mds.get("file_path"))
    ds.setdefault("file_abspath", mds.get("file_abspath"))
    ds.setdefault("metadata_path", mds.get("metadata_path"))
    plan["dataset"] = ds

    # KPIs
    kpis = plan.get("kpis", []) or []
    for k in kpis:
        k["columns"] = _map_to_actual_columns(k.get("columns", []), metadata)
        # derived_fields are freeform expressions; keep as-is
        # ensure label/id present
        k.setdefault("id", re.sub(r"[^a-z0-9_]+", "_", (k.get("label") or "kpi").lower()).strip("_"))
    plan["kpis"] = [k for k in kpis if k.get("columns") or k.get("derived_fields")]

    # Analyses (group_by/metrics/filters are DSL-like; only prune column name lists)
    analyses = plan.get("analyses", []) or []
    for a in analyses:
        a["group_by"] = _map_to_actual_columns(a.get("group_by", []), metadata)
        # metrics/filters are expressions—leave as-is
        a.setdefault("id", re.sub(r"[^a-z0-9_]+", "_", (a.get("label") or "analysis").lower()).strip("_"))
    plan["analyses"] = analyses

    # Visualizations
    vizzes = plan.get("visualizations", []) or []
    for v in vizzes:
        v.setdefault("id", re.sub(r"[^a-z0-9_]+", "_", (v.get("label") or "viz").lower()).strip("_"))
        # data_access fallbacks
        v.setdefault("data_access", {})
        v["data_access"].setdefault("dataset_path", mds.get("file_path") or mds.get("file_abspath"))
        # guess format from file extension if missing
        if "format" not in v["data_access"]:
            ext = (Path(mds.get("file_path") or "").suffix or Path(mds.get("file_abspath") or "").suffix).lower()
            fmt = "csv" if ext in (".csv", ".tsv", ".txt") else ("xlsx" if ext in (".xlsx", ".xls") else ("parquet" if ext in (".parquet", ".pq") else "csv"))
            v["data_access"]["format"] = fmt
        if "encoding" not in v["data_access"]:
            enc = mds.get("file_encoding") or ("excel" if v["data_access"]["format"] == "xlsx" else "utf-8")
            v["data_access"]["encoding"] = enc

        # prune columns_used
        v["columns_used"] = _map_to_actual_columns(v.get("columns_used", []), metadata)

        # spec: x, y, series_by are names; prune y & series_by; keep x if it's a real col
        spec = v.get("spec", {}) or {}
        if "y" in spec:
            spec["y"] = _map_to_actual_columns(spec.get("y", []), metadata)
        if "series_by" in spec:
            spec["series_by"] = _map_to_actual_columns(spec.get("series_by", []), metadata)
        # if x is a column name present, keep it; else leave it (could be derived like MONTH)
        x = spec.get("x")
        if isinstance(x, str) and _normalize_key(x) in {_normalize_key(c) for c in all_cols}:
            spec["x"] = [c for c in all_cols if _normalize_key(c) == _normalize_key(x)][0]
        v["spec"] = spec

        # export path default
        v.setdefault("export", {})
        v["export"].setdefault("suggested_path", f"charts/{v['id']}.html")
        v["export"].setdefault("format", "html")
    plan["visualizations"] = vizzes

    # objectives/notes: keep as-is
    return plan

# ─── Planner function: produce and save plan JSON ───────────────────────────────
def plan_analysis_from_metadata(
    metadata_ref: str | Path,
    objectives: Optional[List[str]] = None,
    out_dir: Path = PLANS_DIR
) -> Tuple[Dict[str, Any], Path]:
    """
    metadata_ref: metadata json path OR dataset file path OR dataset stem
    objectives: optional list of high-level goals (strings)
    """
    metadata, md_path = load_metadata(metadata_ref)
    payload = build_planner_user_payload(metadata, objectives=objectives)

    messages = [
        {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)}
    ]
    resp = llm.invoke(messages)
    plan_raw = _parse_strict_json(resp.content if hasattr(resp, "content") else str(resp))
    plan = _validate_and_prune_plan(plan_raw, metadata)

    # Save plan
    stem = metadata.get("dataset", {}).get("name", "dataset")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{stem}_plan.json"
    out_path.write_text(json.dumps(plan, ensure_ascii=False, indent=2), encoding="utf-8")
    return plan, out_path



In [2]:
# ─── Example usage ──────────────────────────────────────────────────────────────
# 1) With explicit objectives:
plan, plan_path = plan_analysis_from_metadata(
    r"data\sales_data.csv",
    objectives=[
        "Increase sales by understanding MoM and YoY trends",
        "Identify top products and underperformers",
        "Find regional growth opportunities",
        "Analyze deal size mix shift"
    ]
)
print("Saved:", plan_path)
print(json.dumps(plan, indent=2)[:2000] + " ...")

Saved: plans\sales_data_plan.json
{
  "dataset": {
    "name": "sales_data",
    "file_path": "data\\sales_data.csv",
    "file_abspath": "D:\\Autonomous_Dashboard_Creator_Agent\\data\\sales_data.csv",
    "metadata_path": "D:\\Autonomous_Dashboard_Creator_Agent\\metadata\\sales_data_metadata.json"
  },
  "objectives": [
    {
      "text": "Increase sales by understanding MoM and YoY trends",
      "why": "Understanding sales trends helps identify growth opportunities and potential issues."
    },
    {
      "text": "Identify top products and underperformers",
      "why": "Knowing which products drive revenue allows for focused marketing and inventory management."
    },
    {
      "text": "Find regional growth opportunities",
      "why": "Identifying strong and weak regions enables targeted sales strategies."
    },
    {
      "text": "Analyze deal size mix shift",
      "why": "Understanding deal size distribution helps optimize pricing and sales efforts."
    },
    {
      "t

In [ ]:
# 2) Without objectives (auto-suggest by model):
plan, plan_path = plan_analysis_from_metadata(r"data\sales_data.csv")
print("Saved:", plan_path)